In [1]:
!pip install mediapipe==0.10.5 --upgrade --force-reinstall


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.2/467.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.2/355.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

# =========================
# 1) 경로 설정
# =========================
ZIP_PATH   = "/content/drive/MyDrive/버피테스트.zip"   # 드라이브 안 ZIP 경로
IMG_ROOT   = "/content/drive/MyDrive/burpee_images" # ZIP 해제될 폴더
OUTPUT_DIR = "/content/drive/MyDrive/burpee_outputs" # 결과 저장 폴더
CSV_OUT    = f"{OUTPUT_DIR}/burpee_labels.csv"
PKL_OUT    = f"{OUTPUT_DIR}/burpee_dataset.pkl"

import os
os.makedirs(IMG_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 2) ZIP 압축 해제
# =========================
import zipfile
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(IMG_ROOT)
print(f"✅ 압축 해제 완료 → {IMG_ROOT}")

# =========================



✅ 압축 해제 완료 → /content/drive/MyDrive/burpee_images
이미지 개수: 30393


/usr/local/lib/python3.11/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-1162160566.py", line 136, in <cell line: 0>
    img = cv2.imread(p)
          ^^^^^^^^^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'KeyboardInterrupt' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/ultratb.py", line 1101, in get_records
    return _fixed_getinnerframes(etb, number_of_lines_of_context, tb_offset)
           ^^^^^^

TypeError: object of type 'NoneType' has no len()

In [5]:
# 3) 필요 라이브러리 설치
# =========================
import sys
try:
    import cv2, mediapipe as mp, numpy
except:
    !{sys.executable} -q -m pip install mediapipe==0.10.14 opencv-python-headless
import csv, glob, math, pickle
import numpy as np
import mediapipe as mp
import cv2

# =========================
# 4) 유틸 함수 & 라벨 기준
# =========================
mp_pose = mp.solutions.pose

def angle_3pt(a, b, c):
    try:
        ab = (a[0]-b[0], a[1]-b[1])
        cb = (c[0]-b[0], c[1]-b[1])
        dot = ab[0]*cb[0] + ab[1]*cb[1]
        nab = (ab[0]**2 + ab[1]**2) ** 0.5
        ncb = (cb[0]**2 + cb[1]**2) ** 0.5
        cosang = max(-1.0, min(1.0, dot / (nab*ncb + 1e-9)))
        return math.degrees(math.acos(cosang))
    except:
        return float("nan")

def mid_point(p1, p2):
    return ((p1[0]+p2[0])/2.0, (p1[1]+p2[1])/2.0)

def valid_landmarks(landmarks):
    return (landmarks is not None) and (len(landmarks) == 33)

# 버피 기준값
TH = {
    "back_neutral_min": 165.0,
    "elbow_min": 80.0, "elbow_max": 110.0,
    "chest_depth_elbow_max": 100.0,
    "hand_center_tol": 0.12,
    "head_tilt_max": 20.0,
}

def evaluate_burpee_conditions(landmarks):
    NOSE = 0
    L_SHO, R_SHO = 11, 12
    L_ELB, R_ELB = 13, 14
    L_WRI, R_WRI = 15, 16
    L_HIP, R_HIP = 23, 24
    L_ANK, R_ANK = 27, 28

    def get(i):
        lm = landmarks[i]; return (lm.x, lm.y)

    l_sho, r_sho = get(L_SHO), get(R_SHO)
    l_elb, r_elb = get(L_ELB), get(R_ELB)
    l_wri, r_wri = get(L_WRI), get(R_WRI)
    l_hip, r_hip = get(L_HIP), get(R_HIP)
    l_ank, r_ank = get(L_ANK), get(R_ANK)
    nose = get(NOSE)

    chest_center = mid_point(l_sho, r_sho)

    hip_angle_L = angle_3pt(l_sho, l_hip, l_ank)
    hip_angle_R = angle_3pt(r_sho, r_hip, r_ank)
    back_neutral = ((hip_angle_L + hip_angle_R) / 2.0) >= TH["back_neutral_min"]

    elbow_L = angle_3pt(l_sho, l_elb, l_wri)
    elbow_R = angle_3pt(r_sho, r_elb, r_wri)
    elbow_90 = (TH["elbow_min"] <= elbow_L <= TH["elbow_max"]) or (TH["elbow_min"] <= elbow_R <= TH["elbow_max"])

    chest_depth = (min(elbow_L, elbow_R) <= TH["chest_depth_elbow_max"])

    wrists_x = (l_wri[0] + r_wri[0]) / 2.0
    hand_center = (abs(wrists_x - chest_center[0]) <= TH["hand_center_tol"] and
                   max(l_wri[1], r_wri[1]) > max(l_sho[1], r_sho[1]))

    v = (nose[0] - ((l_sho[0]+r_sho[0])/2.0), nose[1] - ((l_sho[1]+r_sho[1])/2.0))
    dot = v[1] * (-1.0)
    nv = (v[0]**2 + v[1]**2) ** 0.5 + 1e-9
    head_tilt = math.degrees(math.acos(max(-1.0, min(1.0, dot/nv))))
    head_neutral = (head_tilt <= TH["head_tilt_max"])

    return back_neutral, elbow_90, chest_depth, hand_center, head_neutral

# =========================
# 5) 이미지 수집
# =========================
exts = {".jpg", ".jpeg", ".png", ".bmp"}
image_paths = sorted(p for p in glob.glob(f"{IMG_ROOT}/**/*.*", recursive=True)
                     if os.path.splitext(p)[1].lower() in exts)
print("이미지 개수:", len(image_paths))
if not image_paths:
    raise SystemExit("❗ IMG_ROOT 경로를 확인하세요.")

# =========================
# 6) 좌표 추출 + 라벨링 → CSV/PKL 저장
# =========================
lm_cols = []
for i in range(33):
    lm_cols += [f"lm{i}_x", f"lm{i}_y", f"lm{i}_z", f"lm{i}_vis"]

header = (["image_path"] + lm_cols +
          ["cond_허리중립","cond_팔꿈치90","cond_가슴이동","cond_손가슴중앙","cond_목중립","label"])

X, y, rows = [], [], []
good = bad = skipped = 0

pose = mp_pose.Pose(static_image_mode=True, model_complexity=1,
                    enable_segmentation=False, min_detection_confidence=0.5)

for p in image_paths:
    img = cv2.imread(p)
    if img is None:
        skipped += 1; continue
    res = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if res.pose_landmarks is None or not valid_landmarks(res.pose_landmarks.landmark):
        skipped += 1; continue

    lms = res.pose_landmarks.landmark
    feats = []
    for lm in lms:
        feats += [lm.x, lm.y, lm.z, lm.visibility]
    X.append(feats)

    c1, c2, c3, c4, c5 = evaluate_burpee_conditions(lms)
    label = 1 if (c1 and c2 and c3 and c4 and c5) else 0
    y.append(label)
    good += int(label == 1); bad += int(label == 0)

    rows.append([p] + feats + [int(c1),int(c2),int(c3),int(c4),int(c5),label])

pose.close()

with open(CSV_OUT, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([header, *rows])

with open(PKL_OUT, "wb") as f:
    pickle.dump({"X": X, "y": y, "header": lm_cols}, f)

print(f"✅ 완료 — 정자세:{good}, 오자세:{bad}, 스킵:{skipped}, 총:{len(image_paths)}")
print("📄 CSV:", CSV_OUT)
print("📦 PKL:", PKL_OUT)

이미지 개수: 30393
✅ 완료 — 정자세:3, 오자세:28264, 스킵:2126, 총:30393
📄 CSV: /content/drive/MyDrive/burpee_outputs/burpee_labels.csv
📦 PKL: /content/drive/MyDrive/burpee_outputs/burpee_dataset.pkl


In [6]:
# ============== 경로 설정 (네 드라이브에 맞게 수정) ==============
DATA_PKL   = "/content/drive/MyDrive/burpee_outputs/burpee_dataset.pkl"  # PKL 우선
DATA_CSV   = "/content/drive/MyDrive/burpee_outputs/burpee_labels.csv"   # PKL 없으면 CSV 사용
MODEL_OUT  = "/content/drive/MyDrive/burpee_outputs/burpee_best_model.pkl"

# ============== 설치/임포트 ==============
import sys, os, pickle, csv
import numpy as np
try:
    import joblib, pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
except:
    !{sys.executable} -q -m pip install scikit-learn joblib pandas
    import joblib, pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

# ============== 데이터 로드 (PKL 우선, 없으면 CSV) ==============
def load_dataset(pkl_path=DATA_PKL, csv_path=DATA_CSV):
    if os.path.isfile(pkl_path):
        with open(pkl_path, "rb") as f:
            d = pickle.load(f)
        X = np.array(d["X"], dtype=np.float32)
        y = np.array(d["y"], dtype=np.int64)
        print(f"Loaded PKL: X={X.shape}, y={y.shape}")
        return X, y

    # CSV에서 로드
    df = pd.read_csv(csv_path)
    # 랜드마크 칼럼만 뽑기
    lm_cols = [c for c in df.columns if c.startswith("lm")]
    X = df[lm_cols].values.astype(np.float32)
    y = df["label"].values.astype(np.int64)
    print(f"Loaded CSV: X={X.shape}, y={y.shape}")
    return X, y

X, y = load_dataset()

# ============== Train/Test split ==============
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ============== 모델들 학습/평가 ==============
results = []

# 1) RandomForest (간편, 성능 안정적)
rf = RandomForestClassifier(
    n_estimators=400, random_state=42, n_jobs=-1, class_weight="balanced_subsample"
)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_te)
acc = accuracy_score(y_te, pred)
cm  = confusion_matrix(y_te, pred)
print("\n[RandomForest]")
print("Accuracy:", acc)
print("Confusion Matrix:\n", cm)
print(classification_report(y_te, pred, digits=4))
results.append(("RandomForest", rf, acc))

# 2) KNN (스케일 필요)
knn = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("knn", KNeighborsClassifier(n_neighbors=7))
])
knn.fit(X_tr, y_tr)
pred = knn.predict(X_te)
acc = accuracy_score(y_te, pred)
cm  = confusion_matrix(y_te, pred)
print("\n[KNN]")
print("Accuracy:", acc)
print("Confusion Matrix:\n", cm)
print(classification_report(y_te, pred, digits=4))
results.append(("KNN", knn, acc))

# 3) MLP (간단한 신경망)
mlp = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("mlp", MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42))
])
mlp.fit(X_tr, y_tr)
pred = mlp.predict(X_te)
acc = accuracy_score(y_te, pred)
cm  = confusion_matrix(y_te, pred)
print("\n[MLP]")
print("Accuracy:", acc)
print("Confusion Matrix:\n", cm)
print(classification_report(y_te, pred, digits=4))
results.append(("MLP", mlp, acc))

# ============== 최고 모델 저장 ==============
best_name, best_model, best_acc = max(results, key=lambda x: x[2])
joblib.dump(best_model, MODEL_OUT)
print(f"\n🏆 Best: {best_name} (acc={best_acc:.4f}) → saved: {MODEL_OUT}")

# ============== (옵션) 단일 이미지 예측 함수 ==============
# 추출 파이프라인 없이 X 벡터(키포인트 132차원)가 있을 때 바로 예측하는 예시.
# (프론트엔드에서 키포인트만 넘겨줄 때 사용)
def predict_from_keypoints(feature_132):
    """feature_132: shape (132,) or (1,132)"""
    m = joblib.load(MODEL_OUT)
    fx = np.array(feature_132, dtype=np.float32).reshape(1, -1)
    return int(m.predict(fx)[0])

# 예: predict_from_keypoints(X_te[0])


Loaded PKL: X=(28267, 132), y=(28267,)

[RandomForest]
Accuracy: 0.9998231340643792
Confusion Matrix:
 [[5653    0]
 [   1    0]]
              precision    recall  f1-score   support

           0     0.9998    1.0000    0.9999      5653
           1     0.0000    0.0000    0.0000         1

    accuracy                         0.9998      5654
   macro avg     0.4999    0.5000    0.5000      5654
weighted avg     0.9996    0.9998    0.9997      5654



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



[KNN]
Accuracy: 0.9998231340643792
Confusion Matrix:
 [[5653    0]
 [   1    0]]
              precision    recall  f1-score   support

           0     0.9998    1.0000    0.9999      5653
           1     0.0000    0.0000    0.0000         1

    accuracy                         0.9998      5654
   macro avg     0.4999    0.5000    0.5000      5654
weighted avg     0.9996    0.9998    0.9997      5654



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



[MLP]
Accuracy: 0.9998231340643792
Confusion Matrix:
 [[5653    0]
 [   1    0]]
              precision    recall  f1-score   support

           0     0.9998    1.0000    0.9999      5653
           1     0.0000    0.0000    0.0000         1

    accuracy                         0.9998      5654
   macro avg     0.4999    0.5000    0.5000      5654
weighted avg     0.9996    0.9998    0.9997      5654



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



🏆 Best: RandomForest (acc=0.9998) → saved: /content/drive/MyDrive/burpee_outputs/burpee_best_model.pkl
